# Raw ReAct Prompt Agent (Notebook Version)


this notebook demonstrates a **raw prompt-driven ReAct loop** with:
- Tool descriptions generated from Python function signatures/docstrings
- Regex-based parsing of `Action` and `Action Input`
- Scratchpad replay each iteration
- Stop token control for tool observation injection


## 1) Setup and Imports

Install dependencies first (outside the notebook if needed):
```bash
pip install langchain langsmith ollama python-dotenv
```

Also ensure Ollama is running and the model is pulled:
```bash
ollama pull qwen3:1.7b
```


In [ ]:
# CHANGE 1: Add re + inspect - parse tool calls from raw text instead of structured JSON.
import inspect
import re

from dotenv import load_dotenv

load_dotenv()

import ollama
from langsmith import traceable

In [ ]:
MAX_ITERATIONS = 10
MODEL = "qwen3:1.7b"

## 2) Tool Definitions

Two simple tools are traced with LangSmith and then placed in a dictionary for dispatch.


In [ ]:
@traceable(run_type="tool")
def get_product_price(product: str) -> float:
    """Look up the price of a product in the catalog."""
    print(f"    >> Executing get_product_price(product='{product}')")
    prices = {"laptop": 1299.99, "headphones": 149.95, "keyboard": 89.50}
    return prices.get(product, 0)


@traceable(run_type="tool")
def apply_discount(price: float, discount_tier: str) -> float:
    """Apply a discount tier to a price and return the final price.
    Available tiers: bronze, silver, gold."""
    print(f"    >> Executing apply_discount(price={price}, discount_tier='{discount_tier}')")
    price = float(price)
    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier, 0)
    return round(price * (1 - discount / 100), 2)

In [ ]:
tools = {
    "get_product_price": get_product_price,
    "apply_discount": apply_discount,
}

## 3) ReAct Prompt Construction

CHANGE 3 from the commit removes JSON schemas and injects tool signatures/docstrings directly into prompt text.


In [ ]:
# Derive plain-text tool descriptions from the decorated functions.
def get_tool_descriptions(tools_dict):
    descriptions = []
    for tool_name, tool_function in tools_dict.items():
        # __wrapped__ bypasses decorator wrappers (e.g., @traceable adds *, config=None).
        original_function = getattr(tool_function, "__wrapped__", tool_function)
        signature = inspect.signature(original_function)
        docstring = inspect.getdoc(tool_function) or ""
        descriptions.append(f"{tool_name}{signature} - {docstring}")
    return "\n".join(descriptions)


tool_descriptions = get_tool_descriptions(tools)
tool_names = ", ".join(tools.keys())


In [ ]:
react_prompt = f"""
STRICT RULES - you must follow these exactly:
1. NEVER guess or assume any product price. You MUST call get_product_price first to get the real price.
2. Only call apply_discount AFTER you have received a price from get_product_price. Pass the exact price returned by get_product_price - do NOT pass a made-up number.
3. NEVER calculate discounts yourself using math. Always use the apply_discount tool.
4. If the user does not specify a discount tier, ask them which tier to use - do NOT assume one.

Answer the following questions as best you can. You have access to the following tools:

{tool_descriptions}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action, as comma separated values
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!

Question: {{question}}
Thought:"""

## 4) Chat Wrapper

CHANGE 4 from the commit: no `tools=` binding in `ollama.chat()`. The model only sees tools via prompt text.


In [ ]:
@traceable(name="Ollama Chat", run_type="llm")
def ollama_chat_traced(model, messages, options):
    return ollama.chat(model=model, messages=messages, options=options)


## 5) Agent Loop

Core behavior from the commit:
- One growing prompt string (`scratchpad`)
- Regex parsing for `Action` + `Action Input`
- Stop token on `\nObservation` so tool results come from Python, not the model


In [ ]:
@traceable(name="Ollama Agent Loop")
def run_agent(question: str):
    print(f"Question: {question}")
    print("=" * 60)

    # CHANGE 5: One prompt string replaces the system/user split.
    prompt = react_prompt.format(question=question)
    scratchpad = ""

    for iteration in range(1, MAX_ITERATIONS + 1):
        print(f"\n--- Iteration {iteration} ---")
        full_prompt = prompt + scratchpad

        # Stop token prevents the model from fabricating Observation values.
        response = ollama_chat_traced(
            model=MODEL,
            messages=[{"role": "user", "content": full_prompt}],
            options={"stop": ["\nObservation"], "temperature": 0},
        )
        output = response.message.content
        print(f"LLM Output:\n{output}")

        print("  [Parsing] Looking for Final Answer in LLM output...")
        final_answer_match = re.search(r"Final Answer:\s*(.+)", output)
        if final_answer_match:
            final_answer = final_answer_match.group(1).strip()
            print(f"  [Parsed] Final Answer: {final_answer}")
            print("\n" + "=" * 60)
            print(f"Final Answer: {final_answer}")
            return final_answer

        # CHANGE 6: Parse tool calls from raw text with regex.
        print("  [Parsing] Looking for Action and Action Input in LLM output...")
        action_match = re.search(r"Action:\s*(.+)", output)
        action_input_match = re.search(r"Action Input:\s*(.+)", output)

        if not action_match or not action_input_match:
            print("  [Parsing] ERROR: Could not parse Action/Action Input from LLM output")
            break

        tool_name = action_match.group(1).strip()
        tool_input_raw = action_input_match.group(1).strip()

        print(f"  [Tool Selected] {tool_name} with args: {tool_input_raw}")

        # Split comma-separated args; strip key= prefix if model returns key=value style.
        raw_args = [x.strip() for x in tool_input_raw.split(",")]
        args = [x.split("=", 1)[-1].strip().strip("'\"") for x in raw_args]

        print(f"  [Tool Executing] {tool_name}({args})...")
        if tool_name not in tools:
            observation = f"Error: Tool '{tool_name}' not found. Available tools: {list(tools.keys())}"
        else:
            observation = str(tools[tool_name](*args))

        print(f"  [Tool Result] {observation}")

        # CHANGE 7: Scratchpad is replayed in full each turn.
        scratchpad += f"{output}\nObservation: {observation}\nThought:"

    print("ERROR: Max iterations reached without a final answer")
    return None


## 6) Run Example

Execute this cell to run the same sample question used in the commit.


In [ ]:
print("Hello LangChain Agent (.bind_tools)!")
print()
result = run_agent("What is the price of a laptop after applying a gold discount?")
result
